In [1]:
#cell 1
#import all the needed tools
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

num_epochs = 20
batch_size = 64
learning_rate = 0.001
num_classes = 10

Using device: cpu


In [ ]:
#cell 2
#for the dataset and the processing

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

#download the training part of CIFAR-10
train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)

#download the testing part of CIFAR-10
test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

#turn datasets into batches
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

#checking the sizes
print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

 74%|███████▍  | 126M/170M [08:03<02:40, 278kB/s]

In [ ]:
#cell 3 - training and evaluation function helpers

def train_one_epoch(model, loader, criterion, optimizer, device):
  model.train()
  running_loss = 0.0

  for images, labels in loader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  return running_loss / len(loader)



def evaluate_model(model, loader, criterion, device):
  model.eval()
  running_loss = 0.0
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for images, labels in loader:
      images = images.to(device)
      labels = labels.to(device)

      outputs = model(images)
      loss = criterion(outputs, labels)
      _, preds = torch.max(outputs, 1)

      running_loss += loss.item()
      all_preds.extend(preds.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

  avg_loss = running_loss / len(loader)
  acc = accuracy_score(all_labels, all_preds)

  return avg_loss, acc, all_labels, all_preds



def train_model(model, train_loader, test_loader, num_epochs, learning_rate, device):
  model = model.to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=learning_rate)

  train_losses = []
  test_losses = []
  test_accuracies = []

  for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, _, _ = evaluate_model(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

  print("\n")
  return model, train_losses, test_losses, test_accuracies




def get_final_metrics(model, loader, device):
  model.eval()
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for images, labels in loader:
      images = images.to(device)
      outputs = model(images)
      _, preds = torch.max(outputs, 1)

      all_preds.extend(preds.cpu().numpy())
      all_labels.extend(labels.numpy())

  acc = accuracy_score(all_labels, all_preds)
  precision = precision_score(all_labels, all_preds, average='macro')
  recall = recall_score(all_labels, all_preds, average='macro')
  f1 = f1_score(all_labels, all_preds, average='macro')

  return acc, precision, recall, f1, all_labels, all_preds

In [ ]:
#cell 4 - Plain CNN-18 & CNN-56 models

class PlainBlock(nn.Module):

  def __init__(self, in_channels, out_channels, stride=1):
    super().__init__()

    self.block = nn.Sequential(
      nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
      nn.BatchNorm2d(out_channels),
      nn.ReLU(inplace=True),
      nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
      nn.BatchNorm2d(out_channels),
      nn.ReLU(inplace=True),
    )

  def forward(self, x):
    return self.block(x)



class PlainCNN18(nn.Module):
  """built manually"""

  def __init__(self, num_classes=10):
    super().__init__()

    self.conv1 = nn.Sequential(
      nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
      nn.BatchNorm2d(64),
      nn.ReLU(inplace=True)
    )

    # 4 stages, each stage has 2 blocks, each block has 2 conv layers
    # total conv layers: 1 first conv + 16 convs in blocks = 17 conv layers
    # final linear layer = 18th learnable layer

    self.layer1 = nn.Sequential(PlainBlock(64, 64), PlainBlock(64, 64))
    self.layer2 = nn.Sequential(PlainBlock(64, 128, stride=2), PlainBlock(128, 128))
    self.layer3 = nn.Sequential(PlainBlock(128, 256, stride=2), PlainBlock(256, 256))
    self.layer4 = nn.Sequential(PlainBlock(256, 512, stride=2), PlainBlock(512, 512))


    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc = nn.Linear(512, num_classes)

  def forward(self, x):
    x = self.conv1(x)
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    x = self.fc(x)
    return x





class PlainCNN56(nn.Module):
  """plain CNN-56 using the CIFAR-style 6n+2 depth idea, where n=9."""

  def __init__(self, num_classes=10):
    super().__init__()
    n = 9

    self.conv1 = nn.Sequential(
      nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False),
      nn.BatchNorm2d(16),
      nn.ReLU(inplace=True),
    )

    self.layer1 = self._make_layer(16, 16, n, stride=1)
    self.layer2 = self._make_layer(16, 32, n, stride=2)
    self.layer3 = self._make_layer(32, 64, n, stride=2)

    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc = nn.Linear(64, num_classes)

  def _make_layer(self, in_channels, out_channels, num_blocks, stride):
    layers = [PlainBlock(in_channels, out_channels, stride=stride)]
    for _ in range(1, num_blocks):
      layers.append(PlainBlock(out_channels, out_channels, stride=1))
    return nn.Sequential(*layers)

  def forward(self, x):
    x = self.conv1(x)
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    return self.fc(x)




In [ ]:
#cell 5 - ResNet-18 & ResNet-56 models
def create_resnet18(num_classes=10):
  """TorchVision ResNet-18"""
  resnet18 = torchvision.models.resnet18(weights=None)
  resnet18.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False) #change the first convolution layer to better fit the dataset.
  resnet18.maxpool = nn.Identity()
  resnet18.fc = nn.Linear(resnet18.fc.in_features, num_classes) #change the final output layer so the model predicts 10 classes.
  return resnet18



class ResidualBlock(nn.Module):
  """Residual block: Conv-BN-ReLU-Conv-BN + shortcut."""

  def __init__(self, in_channels, out_channels, stride=1):
    super().__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU(inplace=True)

    self.shortcut = nn.Sequential()
    if stride != 1 or in_channels != out_channels:
      self.shortcut = nn.Sequential(
      nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False), nn.BatchNorm2d(out_channels),
      )

  def forward(self, x):
    identity = self.shortcut(x)

    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)

    out = self.conv2(out)
    out = self.bn2(out)

    out = out + identity
    out = self.relu(out)

    return out


class ResNet56(nn.Module):
  """ResNet-56 implemented manually for CIFAR-10, where depth = 6n+2 and n=9."""

  def __init__(self, num_classes=10):
    super().__init__()
    n = 9

    self.conv1 = nn.Sequential(
      nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
    )

    self.layer1 = self._make_layer(16, 16, n, stride=1)
    self.layer2 = self._make_layer(16, 32, n, stride=2)
    self.layer3 = self._make_layer(32, 64, n, stride=2)

    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc = nn.Linear(64, num_classes)

  def _make_layer(self, in_channels, out_channels, num_blocks, stride):
    layers = [ResidualBlock(in_channels, out_channels, stride=stride)]

    for _ in range(1, num_blocks):
      layers.append(ResidualBlock(out_channels, out_channels, stride=1))

    return nn.Sequential(*layers)

  def forward(self, x):
    x = self.conv1(x)
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    return self.fc(x)



In [ ]:
#cell 6 - train 18
experiments = {}

#plain CNN-18
plain_cnn18 = PlainCNN18(num_classes=num_classes)
plain_cnn18, plain18_train_losses, plain18_test_losses, plain18_test_accuracies = train_model(
    plain_cnn18, train_loader, test_loader, num_epochs, learning_rate, device
)

experiments["Plain CNN-18"] = {
    "model": plain_cnn18,
    "train_losses": plain18_train_losses,
    "test_losses": plain18_test_losses,
    "test_accuracies": plain18_test_accuracies,
}


#ResNet-18
resnet18 = create_resnet18(num_classes=num_classes)
resnet18, resnet18_train_losses, resnet18_test_losses, resnet18_test_accuracies = train_model(
  resnet18, train_loader, test_loader, num_epochs, learning_rate, device
)
experiments["ResNet-18"] = {
  "model": resnet18,
  "train_losses": resnet18_train_losses,
  "test_losses": resnet18_test_losses,
  "test_accuracies": resnet18_test_accuracies,
}

Epoch [1/20] | Train Loss: 1.7520 | Test Loss: 1.6563 | Test Acc: 0.3879
Epoch [2/20] | Train Loss: 1.4430 | Test Loss: 1.2380 | Test Acc: 0.5508


In [ ]:
#cell 7 - train 56
# Plain CNN-56
plain_cnn56 = PlainCNN56(num_classes=num_classes)
plain_cnn56, plain56_train_losses, plain56_test_losses, plain56_test_accuracies = train_model(
  plain_cnn56, train_loader, test_loader, num_epochs, learning_rate, device
)
experiments["Plain CNN-56"] = {
  "model": plain_cnn56,
  "train_losses": plain56_train_losses,
  "test_losses": plain56_test_losses,
  "test_accuracies": plain56_test_accuracies,
}

# ResNet-56
resnet56 = ResNet56(num_classes=num_classes)
resnet56, resnet56_train_losses, resnet56_test_losses, resnet56_test_accuracies = train_model(
  resnet56, train_loader, test_loader, num_epochs, learning_rate, device
)
experiments["ResNet-56"] = {
  "model": resnet56,
  "train_losses": resnet56_train_losses,
  "test_losses": resnet56_test_losses,
  "test_accuracies": resnet56_test_accuracies,
}


In [ ]:
#cell 8 - final metrics table

results = []

for model_name, data in experiments.items():
    acc, precision, recall, f1, labels, preds = get_final_metrics(
        data["model"], test_loader, device
    )

    data["labels"] = labels
    data["preds"] = preds

    results.append({
        "Model": model_name,
        "Epochs": num_epochs,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
    })

results_df = pd.DataFrame(results)
print("\nFinal Results Table")
print(results_df.to_string(index=False))
print("\n\n\n\n\n\n\n\n\n")





#plots
plt.figure(figsize=(9, 5))
plt.plot(
    experiments["Plain CNN-18"]["test_accuracies"],
    label="Plain CNN-18",
    color="red",
    marker="o"
)
plt.plot(
    experiments["ResNet-18"]["test_accuracies"],
    label="ResNet-18",
    color="blue",
    marker="o"
)
plt.plot(
    experiments["Plain CNN-56"]["test_accuracies"],
    label="Plain CNN-56",
    color="orange",
    marker="o"
)
plt.plot(
    experiments["ResNet-56"]["test_accuracies"],
    label="ResNet-56",
    color="green",
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("Test Accuracy")
plt.title("Test Accuracy Comparison for All Models")
plt.legend()
plt.grid(True)
plt.show()
print("\n\n\n\n")

plt.figure(figsize=(9, 5))
epochs = range(1, num_epochs + 1)
plt.plot(
    epochs,
    experiments["Plain CNN-18"]["train_losses"],
    label="Plain CNN-18",
    color="red",
    marker="o"
)
plt.plot(
    epochs,
    experiments["ResNet-18"]["train_losses"],
    label="ResNet-18",
    color="blue",
    marker="o"
)
plt.plot(
    epochs,
    experiments["Plain CNN-56"]["train_losses"],
    label="Plain CNN-56",
    color="orange",
    marker="o"
)
plt.plot(
    epochs,
    experiments["ResNet-56"]["train_losses"],
    label="ResNet-56",
    color="green",
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Comparison for All Models")
plt.legend()
plt.grid(True)
plt.show()
print("\n\n\n\n")



def plot_training_loss(model_a, model_b, title):
    plt.figure(figsize=(8, 5))
    plt.plot(experiments[model_a]["train_losses"], label=f"{model_a} Train Loss")
    plt.plot(experiments[model_b]["train_losses"], label=f"{model_b} Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.show()

print("\n\n\n\n")

def plot_test_accuracy(model_a, model_b, title):
    plt.figure(figsize=(8, 5))
    plt.plot(experiments[model_a]["test_accuracies"], label=f"{model_a} Test Accuracy")
    plt.plot(experiments[model_b]["test_accuracies"], label=f"{model_b} Test Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.show()

print("\n\n\n\n")

def plot_confusion_matrix(model_name):
    cm = confusion_matrix(experiments[model_name]["labels"], experiments[model_name]["preds"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(cmap="Blues", xticks_rotation=45)
    plt.title(f"{model_name} Confusion Matrix")
    plt.show()


# 18-layer comparison plots
plot_confusion_matrix("Plain CNN-18")
print("\n\n")
plot_confusion_matrix("ResNet-18")

print("\n\n\n\n")

# 56-layer comparison plots
plot_confusion_matrix("Plain CNN-56")
print("\n\n")
plot_confusion_matrix("ResNet-56")